# Binary Discrete Diffusion — larger Sudoku run (Colab, GPU)

Paper: [arXiv:2610.28543](https://arxiv.org/abs/2610.28543) (Casado Noguerales et al.). **No official code** was found; this notebook scales the **independent toy** in `apps/binary-discrete-diffusion-sudoku/` (not a reproduction of the authors' 43M DiT / Sudoku-Extreme setup).

> ⚠️ **GPU budget needs owner approval.** This notebook is prepared but **not executed**. The CPU clear-experiment in the app folder never needs a GPU. Use a free Colab T4 only after the owner agrees; anything paid (A100/L4, Colab Pro) needs explicit approval first.

What you get: a larger transformer, more training steps, and a low-clue sweep closer in spirit to the paper's hard Sudoku regime — still a pedagogical re-implementation.

In [ ]:
# 0) Check the GPU Colab gave you (Runtime > Change runtime type > GPU)
!nvidia-smi

In [ ]:
# 1) Clone this monorepo and install the app deps (Colab ships CUDA torch)
!git clone --depth 1 https://github.com/Ryukijano/tech-demos.git
%cd tech-demos/apps/binary-discrete-diffusion-sudoku
!python -m pip install -q -r requirements.txt

In [ ]:
# 2) Scale the pre-registered toy: bigger trunk, more steps, harder clues
# Edit experiments/exp1_cpu_sudoku/config.yaml or override below.
import yaml
from pathlib import Path
cfg_path = Path('experiments/exp1_cpu_sudoku/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['model']['d_model'] = 256
cfg['model']['n_layers'] = 6
cfg['model']['n_heads'] = 8
cfg['training']['steps'] = 20000
cfg['training']['batch_size'] = 64
cfg['data']['n_clues_train'] = 36
cfg['data']['n_clues_eval'] = 36
cfg['data']['n_train_puzzles'] = 2048
cfg['data']['n_eval_puzzles'] = 256
cfg['sampling']['nfe'] = 64
cfg['runtime']['device'] = 'cuda'
cfg_path.write_text(yaml.safe_dump(cfg))
print('updated config for larger GPU run (still a toy, not Sudoku-Extreme)')

In [ ]:
# 3) Run the comparison (seeds 0/1/2) — expect tens of minutes on a T4
import os
os.environ['PYTHONPATH'] = 'src'
!python src/run_experiment.py --skip-breakpoints
!python src/analyze.py
from IPython.display import Image, display, Markdown
display(Markdown(Path('experiments/exp1_cpu_sudoku/results.md').read_text()))
display(Image('experiments/exp1_cpu_sudoku/plot.png'))

## Notes

- Still **not** the paper's architecture, dataset, or sampler.
- If/when the authors release code, prefer that for any claim about Table 1.
- Do not spend paid GPU without owner approval.